load checkpoint and resume training for interrupted runs

In [1]:
import torch
import wandb
import numpy as np

In [ ]:
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from model.clip import model as clip_model, processor as clip_processor
from fairface_vit import FairFaceViT

In [3]:
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import (
    get_train_transform,
    get_age_transform,
    get_val_transform,
)

In [4]:
from training.train import train_loop, test_loop
from training.losses import (
    get_age_weights,
    get_loss_function,
)

In [ ]:
import os
model_name = 'clip' 

os.makedirs(f"../checkpoints/{model_name}", exist_ok=True)

epochs = 20

batch_size = 16
learning_rate = 1e-4
weight_decay = 1e-2

age_dropout1 = 0.15
age_hidden_dim = 384

loss_weights = {
    "gender": 1,
    "age": 1,
    "race": 1,
}

In [ ]:
# offline-run-20260717_234704-clip-vit-base-patch16
wandb.init(
    project="fairface-vit",
    name="clip-vit-base-patch16",
    id="clip-vit-base-patch16",   # Same ID as the original run
    
    resume="must",
    mode="offline",
    
    config={
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "age_dropout1": age_dropout1,
        "age_hidden_dim": age_hidden_dim,
        "weight_decay": weight_decay,
        "optimizer": "AdamW",
        "model": "clip-vit-base-patch16",
        "loss_weights": loss_weights,
    }
)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

fairface_clip_model = FairFaceViT(
    clip_model,
    age_dropout1,
    age_hidden_dim,
)
fairface_clip_model.to(device)

FairFaceViT(
  (backbone): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)

In [ ]:
train_set, val_set = get_dataset(
    get_train_transform(clip_processor.image_processor),
    get_age_transform(clip_processor.image_processor),
    get_val_transform(clip_processor.image_processor),
)

train_dataloader, val_dataloader = get_dataloaders(
    train_set,
    val_set,
    batch_size=batch_size,
    num_workers=0
)

In [11]:
age_labels = np.array(train_set.dataset["age"])

age_weights = get_age_weights(
    age_labels,
    num_classes=9,
    device=device,
)

loss_funcs = get_loss_function(age_weights)

In [12]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fairface_clip_model.parameters()),
    lr=learning_rate,
    weight_decay=weight_decay,
)

In [13]:
checkpoint = torch.load(
    f"../checkpoints/{model_name}/{model_name}-checkpoint.pt",
    map_location=device
)
print(checkpoint.keys())
print(checkpoint["epoch"])

dict_keys(['gender_head', 'age_head', 'race_head', 'optimizer_state_dict', 'epoch', 'best_acc', 'patience', 'patience_counter', 'min_delta'])
7


dict_keys(['gender_head', 'age_head', 'race_head', 'optimizer_state_dict', 'epoch', 'best_acc'])
19


In [14]:
fairface_clip_model.gender.load_state_dict(
    checkpoint["gender_head"]
)

fairface_clip_model.age.load_state_dict(
    checkpoint["age_head"]
)

fairface_clip_model.race.load_state_dict(
    checkpoint["race_head"]
)


optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)


best_acc = checkpoint["best_acc"]
start_epoch = checkpoint["epoch"] + 1

patience = checkpoint['patience']
patience_counter = checkpoint['patience_counter']
min_delta = checkpoint['min_delta']
# patience = 5
# patience_counter = 4
# min_delta = 0.002

import random

torch.set_rng_state(checkpoint["torch_rng_state"])
torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
np.random.set_state(checkpoint["numpy_rng_state"])
random.setstate(checkpoint["python_rng_state"])


print(
    f"Loaded checkpoint.pt | "
    f"Resume from epoch {start_epoch} | "
    f"Best acc={best_acc:.4f}"
)

KeyError: 'torch_rng_state'

In [ ]:


for epoch in range(start_epoch, epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_task_loss, train_metrics = train_loop(
        train_dataloader, fairface_clip_model, loss_funcs, 
        loss_weights, optimizer, device, epoch+1, epochs
    )

    val_loss, val_task_loss, metrics, subgroup_metrics = test_loop(
        val_dataloader,fairface_clip_model, loss_funcs, 
        loss_weights, device, epoch+1, epochs
    )

    log_dict = {

        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        
        # losses
        "train/loss": train_loss,
        "val/loss": val_loss,

        "train/gender_loss": train_task_loss["gender"],
        "train/age_loss": train_task_loss["age"],
        "train/race_loss": train_task_loss["race"],

        "val/gender_loss": val_task_loss["gender"],
        "val/age_loss": val_task_loss["age"],
        "val/race_loss": val_task_loss["race"],
    }

   # train metrics
    for task, values in train_metrics.items():
        for metric_name, value in values.items():
            log_dict[f"train/{task}/{metric_name}"] = value

    # val metrics
    for task, values in metrics.items():
        for metric_name, value in values.items():
            log_dict[f"val/{task}/{metric_name}"] = value

    # subgroup accuracy
    for group_name, values in subgroup_metrics.items():
        for subgroup, acc in values.items():
            log_dict[f"subgroup/{group_name}/{subgroup}"] = acc

    
    current_acc = (
        metrics["gender"]["accuracy"]
        + metrics["age"]["accuracy"]
        + metrics["race"]["accuracy"]
    ) / 3

    log_dict["val/avg_accuracy"] = current_acc

    wandb.log(log_dict)

    if current_acc > best_acc + min_delta:

        best_acc = current_acc
        patience_counter = 0

        torch.save(
            {
                "gender_head": fairface_clip_model.gender.state_dict(),
                "age_head": fairface_clip_model.age.state_dict(),
                "race_head": fairface_clip_model.race.state_dict(),
            },
            f"../checkpoints/{model_name}/{model_name}-best_heads.pt",
        )

        print(f"Saved {model_name}-best_heads.pt | avg accuracy={best_acc:.4f}")

    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

    torch.save(
        {
            "gender_head": fairface_clip_model.gender.state_dict(),
            "age_head": fairface_clip_model.age.state_dict(),
            "race_head": fairface_clip_model.race.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),

            "epoch": epoch,
            "best_acc": best_acc,

            "patience": patience,
            "patience_counter": patience_counter,
            "min_delta": min_delta,
        },
        f"../checkpoints/{model_name}/{model_name}-checkpoint.pt",
    )

    if patience_counter >= patience:
        print(f"\nEarly stopping after {epoch+1} epochs.")
        break   


Epoch 8/20


 avg loss:        1.6611
 avg age loss:    0.82
 avg gender loss: 0.13
 avg race loss:   0.72

 gender: acc=0.951  f1=0.947
 age:    acc=0.604  f1=0.610  mae=0.444
 race:   acc=0.731  f1=0.724


ValueError: too many values to unpack (expected 2)

In [ ]:
wandb.finish()

epoch,▁█
lr,▁▁
subgroup/age by gender/0,▁█
subgroup/age by gender/1,█▁
subgroup/age by race/0,█▁
subgroup/age by race/1,▁█
subgroup/age by race/2,▁█
subgroup/age by race/3,▁█
subgroup/age by race/4,▁█
subgroup/age by race/5,█▁
+52,...
